# UTS Business Intelligence: Integrasi Data Mining dan BI pada Data Transaksi Toko
**Nama:** Aprilia Katarina Ua | **NPM:** 51220076 | **Kelas:** B | **Semester:** IX
**Mata kuliah:** Business Intelligence (TIF46212), Universitas Timor

Notebook ini memuat EDA, preprocessing, feature engineering, Apriori, K-Means, dan ekspor hasil.
Letakkan file `P8_UTS_Business_Intelligence.xlsx` di folder yang sama dengan notebook ini (ubah `FILE` jika namanya berbeda).

In [ ]:
import json, itertools
from collections import Counter
from itertools import combinations
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
FILE = 'P8_UTS_Business_Intelligence.xlsx'
raw = pd.read_excel(FILE, sheet_name='Data_Raw')
prod = pd.read_excel(FILE, sheet_name='Data_Produk')
R = {}  # ringkasan hasil

## Tahap 1. Business Understanding
**Masalah utama:** data transaksi hanya dipakai untuk laporan rutin; pola pembelian, kombinasi produk, dan segmen pelanggan belum dimanfaatkan, sementara data masih kotor.

**Pertanyaan bisnis:**
1. Produk dan kategori apa yang menyumbang penjualan terbesar, dan bagaimana tren bulanannya?
2. Produk apa yang sering dibeli bersamaan sehingga layak dibundel atau ditempatkan berdekatan?
3. Kelompok pelanggan apa yang ada, dan mana yang berisiko berhenti membeli?

Hasil analisis membantu manajemen menaikkan penjualan (cross-selling, bundling) dan mempertahankan pelanggan (program loyalitas, kampanye win-back).

## Tahap 1. Exploratory Data Analysis (data mentah)

In [ ]:
print('Baris, kolom:', raw.shape)
print(raw.dtypes)
print('Rentang waktu:', raw.Transaction_Date.min(), 's.d.', raw.Transaction_Date.max())
print('Transaksi unik:', raw.Transaction_ID.nunique(), '| pelanggan unik:', raw.Customer_ID.nunique(), '| produk:', raw.Product_Name.nunique())
print('Rata-rata Quantity (mentah, terpengaruh outlier):', round(raw.Quantity.mean(), 2))
print('Total Line_Total (mentah):', raw.Line_Total.sum())
miss = raw.isna().sum(); print(miss[miss > 0])
print('Duplikat baris penuh:', raw.duplicated().sum())
print('Kategori tidak konsisten:', (~raw.Category.isin(prod.Category.unique())).sum())
print('Quantity <=0 atau >50:', ((raw.Quantity <= 0) | (raw.Quantity > 50)).sum(), '| Unit_Price <=0 atau >200000:', ((raw.Unit_Price <= 0) | (raw.Unit_Price > 200000)).sum())
R['eda_raw'] = dict(rows=len(raw), cols=raw.shape[1], tx=int(raw.Transaction_ID.nunique()), cust=int(raw.Customer_ID.nunique()), dup=int(raw.duplicated().sum()), miss={k: int(v) for k, v in miss[miss > 0].items()})

## Tahap 2. Data Preprocessing
Keputusan dan alasan:
- **Duplikat penuh dihapus** karena nota yang tercatat dua kali menggandakan penjualan.
- **Customer_ID kosong diisi dari baris lain pada nota yang sama** (satu nota milik satu pelanggan), sehingga tidak ada data yang dibuang.
- **Product_ID dan kategori diambil dari Data_Produk** berdasarkan nama produk, sebagai sumber acuan yang konsisten.
- **Teks pembayaran diseragamkan** (huruf dan spasi).
- **Quantity tidak wajar (kosong, <=0, >50) dihitung ulang** dari Line_Total / (harga x (1 - diskon)); nilai 999 jelas kesalahan input, bukan pembelian grosir.
- **Unit_Price kosong atau tidak wajar diganti harga referensi**, karena semua harga yang sah di data sama dengan harga referensi.
- **Line_Total dihitung ulang** = Quantity x Unit_Price x (1 - Discount_Rate).

In [ ]:
d = raw.drop_duplicates().copy()
n_dup = len(raw) - len(d)
pid = dict(zip(prod.Product_Name, prod.Product_ID)); ref = prod.set_index('Product_Name')
d['Product_ID'] = d.Product_Name.map(pid)
d['Category'] = d.Product_Name.map(ref.Category)
d['Payment_Method'] = d.Payment_Method.str.strip().str.upper().replace({'TUNAI': 'Tunai', 'DEBIT': 'Debit', 'TRANSFER': 'Transfer'})
refp = d.Product_Name.map(ref.Reference_Unit_Price_Rp)
price_ok = (d.Unit_Price > 0) & (d.Unit_Price <= 200000)
n_price = int((~price_ok).sum())
d['Unit_Price'] = np.where(price_ok, d.Unit_Price, refp)
q = d.Quantity.copy(); bad = q.isna() | (q <= 0) | (q > 50)
est = (d.Line_Total / (d.Unit_Price * (1 - d.Discount_Rate))).round()
fix = bad & est.between(1, 20); q[fix] = est[fix]
q[bad & ~fix] = d.groupby('Product_Name').Quantity.transform(lambda s: s[(s > 0) & (s <= 50)].median())[bad & ~fix].round()
n_qty = int(bad.sum()); d['Quantity'] = q
d['Line_Total'] = (d.Quantity * d.Unit_Price * (1 - d.Discount_Rate)).round()
n_dup2 = int(d.duplicated(['Transaction_ID', 'Product_Name']).sum())
d = d.drop_duplicates(['Transaction_ID', 'Product_Name'])
n_cid = int(d.Customer_ID.isna().sum())
d['Customer_ID'] = d.groupby('Transaction_ID').Customer_ID.transform(lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan)
d['Transaction_Date'] = pd.to_datetime(d.Transaction_Date)
d['Month'] = d.Transaction_Date.dt.month
log = pd.DataFrame([
    ['Duplikat baris penuh', n_dup, 'dihapus'], ['Duplikat produk dalam satu nota', n_dup2, 'dihapus'],
    ['Customer_ID kosong', n_cid, 'diisi dari nota yang sama'], ['Quantity kosong/tidak wajar', n_qty, 'dihitung ulang'],
    ['Unit_Price kosong/tidak wajar', n_price, 'diganti harga referensi'], ['Baris mentah', len(raw), ''], ['Baris bersih', len(d), '']],
    columns=['Masalah', 'Jumlah', 'Penanganan'])
print(log.to_string(index=False))
print('Sisa kosong pada kolom analisis:', d[['Customer_ID', 'Product_ID', 'Quantity', 'Unit_Price', 'Line_Total']].isna().sum().sum())
R['clean'] = dict(rows=len(d), tx=int(d.Transaction_ID.nunique()), cust=int(d.Customer_ID.nunique()), total=float(d.Line_Total.sum()), qty_mean=float(d.Quantity.mean()), qty_fixed=n_qty, price_fixed=n_price, dup2=n_dup2, cid_fixed=n_cid)

## Tahap 1. Visualisasi dan temuan (data bersih)

In [ ]:
tx = d.groupby('Transaction_ID').agg(cust=('Customer_ID', 'first'), date=('Transaction_Date', 'first'), amount=('Line_Total', 'sum'), basket_size=('Product_Name', 'nunique')).reset_index()
tx['month'] = tx.date.dt.month
monthly = d.groupby('Month').Line_Total.sum()
topp = d.groupby('Product_Name').Line_Total.sum().sort_values(ascending=False)
cat = d.groupby('Category').Line_Total.sum().sort_values(ascending=False)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
monthly.plot(ax=ax[0], marker='o', color='#f28a1e'); ax[0].set_title('Tren penjualan bulanan 2025'); ax[0].set_xlabel('Bulan'); ax[0].set_ylabel('Rp')
topp.head(10)[::-1].plot.barh(ax=ax[1], color='#0f7f8c'); ax[1].set_title('10 produk dengan penjualan tertinggi')
plt.tight_layout(); plt.show()
print('Rata-rata basket_size:', round(tx.basket_size.mean(), 2), '| rata-rata nilai transaksi:', round(tx.amount.mean()))
R['eda'] = dict(monthly={int(k): float(v) for k, v in monthly.items()}, top10={k: float(v) for k, v in topp.head(10).items()}, cat={k: float(v) for k, v in cat.items()}, basket=float(tx.basket_size.mean()), avg_tx=float(tx.amount.mean()), pay=d.drop_duplicates('Transaction_ID').Payment_Method.value_counts().to_dict())

## Tahap 2. Feature Engineering: RFM, basket_size, matriks transaksi, normalisasi

In [ ]:
ref_date = pd.Timestamp('2026-01-01')  # sehari setelah transaksi terakhir, agar Recency >= 1
rfm = tx.groupby('cust').agg(Recency=('date', lambda s: (ref_date - s.max().normalize()).days), Frequency=('amount', 'size'), Monetary=('amount', 'sum')).reset_index()
print(rfm.describe().round(1))
# Matriks transaksi (one-hot) untuk Apriori
baskets = d.groupby('Transaction_ID').Product_Name.apply(lambda s: sorted(set(s))).tolist()
basket_matrix = pd.DataFrame([{p: True for p in b} for b in baskets]).fillna(False)
print('Matriks transaksi:', basket_matrix.shape)
# Log transform + standardisasi (Z-score) agar skala Monetary tidak mendominasi jarak K-Means
X = np.log1p(rfm[['Recency', 'Frequency', 'Monetary']]); Xs = (X - X.mean()) / X.std(ddof=0)
# Strategi data besar: agregasi per pelanggan (sudah dilakukan), sampling, dan chunking saat membaca file.

## Tahap 3. Apriori (implementasi sendiri, itemset sampai ukuran 3)
**Minimum support 1%** dipilih karena ada 30 produk dan rata-rata hanya 3 produk per nota, sehingga setiap pasangan jarang muncul; 1% setara 60 nota, cukup untuk dianggap pola. **Minimum confidence 20%** agar aturan masih berarti tetapi tidak terlalu ketat. **Kriteria peringkat:** lift tertinggi, lalu confidence.

In [ ]:
def apriori(baskets, min_sup):
    N = len(baskets); cnt = Counter(i for b in baskets for i in b)
    prev = {frozenset([i]) for i, c in cnt.items() if c / N >= min_sup}
    freq = {s: cnt[next(iter(s))] for s in prev}; k = 2
    while prev:
        cands = {a | b for a in prev for b in prev if len(a | b) == k and all(frozenset(s) in prev for s in combinations(a | b, k - 1))}
        cc = Counter()
        for b in baskets:
            bs = set(b)
            for c in cands:
                if c <= bs: cc[c] += 1
        prev = {c for c, n in cc.items() if n / N >= min_sup}
        freq.update({c: cc[c] for c in prev}); k += 1
    return freq, N
def rules_from(freq, N, min_conf):
    out = []
    for s, n in freq.items():
        if len(s) < 2: continue
        for r in range(1, len(s)):
            for a in combinations(s, r):
                a = frozenset(a); b = s - a
                conf = n / freq[a]; out.append((', '.join(sorted(a)), ', '.join(sorted(b)), n / N, conf, conf / (freq[b] / N)))
    return pd.DataFrame(out, columns=['antecedent', 'consequent', 'support', 'confidence', 'lift']).sort_values(['lift', 'confidence'], ascending=False)
freq, N = apriori(baskets, 0.01)
rules = rules_from(freq, N, 0.20); rules = rules[rules.confidence >= 0.20].reset_index(drop=True)
print('Frequent itemset per ukuran:', Counter(len(s) for s in freq))
top_rules = rules.head(10); print(top_rules.round(3).to_string(index=False))
R['rules'] = top_rules.round(4).values.tolist(); R['n_rules'] = len(rules); R['freq_sizes'] = dict(Counter(len(s) for s in freq))

In [ ]:
# Visualisasi network graph aturan asosiasi (tata letak melingkar)
nodes = sorted({x for r in top_rules.itertuples() for x in (r.antecedent.split(', ') + r.consequent.split(', '))})
pos = {n: (np.cos(2 * np.pi * i / len(nodes)), np.sin(2 * np.pi * i / len(nodes))) for i, n in enumerate(nodes)}
plt.figure(figsize=(7, 7))
for r in top_rules.itertuples():
    for a in r.antecedent.split(', '):
        for b in r.consequent.split(', '):
            plt.plot([pos[a][0], pos[b][0]], [pos[a][1], pos[b][1]], color='#f28a1e', lw=r.lift, alpha=.6)
for n, (x, y) in pos.items(): plt.scatter(x, y, s=120, color='#0f7f8c'); plt.text(x, y + .06, n, ha='center', fontsize=8)
plt.axis('off'); plt.title('Network graph 10 aturan asosiasi teratas'); plt.show()

## Tahap 4. K-Means: Elbow dan Silhouette

In [ ]:
ks = range(2, 7); inertia, sil = [], []
for k in ks:
    km = KMeans(k, n_init=10, random_state=1).fit(Xs); inertia.append(km.inertia_); sil.append(silhouette_score(Xs, km.labels_))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(list(ks), inertia, marker='o'); ax[0].set_title('Elbow (inertia)'); ax[1].bar(list(ks), sil); ax[1].set_title('Silhouette score')
plt.show(); print(dict(zip(ks, np.round(sil, 3))))
R['kmeans_k'] = {int(k): [float(i), float(s)] for k, i, s in zip(ks, inertia, sil)}

**Pemilihan k = 3.** Silhouette tertinggi ada pada k = 2, tetapi k = 2 hanya memisahkan pelanggan aktif dan tidak aktif. k = 3 memberi skor yang masih wajar dan tiga segmen yang dapat ditindaklanjuti (Loyal, Potential, Churn Risk).

In [ ]:
km = KMeans(3, n_init=10, random_state=1).fit(Xs); rfm['cluster'] = km.labels_
prof = rfm.groupby('cluster').agg(Pelanggan=('cust', 'size'), Recency=('Recency', 'mean'), Frequency=('Frequency', 'mean'), Monetary=('Monetary', 'mean')).round(1)
loyal = prof.Frequency.idxmax(); churn = prof.drop(loyal).Recency.idxmax(); pot = [c for c in prof.index if c not in (loyal, churn)][0]
names = {loyal: 'Loyal Customers', churn: 'Churn Risk', pot: 'Potential Customers'}
rfm['Segmen'] = rfm.cluster.map(names); prof['Segmen'] = prof.index.map(names); print(prof)
colors = {'Loyal Customers': '#0f7f8c', 'Churn Risk': '#d6453d', 'Potential Customers': '#3b6fd4'}
plt.figure(figsize=(7, 4.5))
for s, g in rfm.groupby('Segmen'): plt.scatter(g.Recency + np.random.rand(len(g)) * 2, g.Frequency + np.random.rand(len(g)) * .3, s=10, c=colors[s], label=s, alpha=.6)
plt.xlabel('Recency (hari)'); plt.ylabel('Frequency'); plt.legend(); plt.title('Segmentasi pelanggan K-Means (k=3)'); plt.show()
R['segments'] = prof.reset_index().to_dict('records')

## Bonus: integrasi Apriori dan K-Means (aturan teratas per segmen)

In [ ]:
seg_of = rfm.set_index('cust').Segmen; bonus = {}
tx['Segmen'] = tx.cust.map(seg_of); dd = d.merge(tx[['Transaction_ID', 'Segmen']], on='Transaction_ID')
for s in names.values():
    ids = set(tx[tx.Segmen == s].Transaction_ID)
    bs = dd[dd.Transaction_ID.isin(ids)].groupby('Transaction_ID').Product_Name.apply(lambda x: sorted(set(x))).tolist()
    f, n = apriori(bs, 0.01); r = rules_from(f, n, 0.2); r = r[r.confidence >= .2]
    top = dd[dd.Transaction_ID.isin(ids)].groupby('Product_Name').Line_Total.sum().sort_values(ascending=False).head(3)
    bonus[s] = dict(tx=len(bs), rule=r.head(1).round(3).values.tolist(), top=list(top.index))
    print(s, bonus[s])
R['bonus'] = bonus

## Tahap 6. Estimasi dampak (asumsi, bukan hasil data)

In [ ]:
c = rfm[rfm.Segmen == 'Churn Risk']; avg_tx_churn = c.Monetary.sum() / c.Frequency.sum()
est = len(c) * 0.10 * avg_tx_churn
print('Pelanggan Churn Risk:', len(c), '| rata-rata nilai per transaksi:', round(avg_tx_churn), '| estimasi tambahan penjualan (asumsi 10% kembali membeli 1x):', round(est))
R['impact'] = dict(churn=len(c), avg_tx=float(avg_tx_churn), est=float(est))

## Ekspor luaran (dataset bersih, log preprocessing, RFM, aturan)

In [ ]:
d.to_csv('data_bersih_preprocessing.csv', index=False)
log.to_csv('dokumentasi_preprocessing.csv', index=False)
rfm.to_csv('rfm_segmen_pelanggan.csv', index=False)
rules.to_csv('aturan_asosiasi_apriori.csv', index=False)
json.dump(R, open('hasil_ringkasan.json', 'w'), ensure_ascii=False, indent=1, default=str)
print('Selesai. Baris data bersih:', len(d))